# 02 · Features y preparación sin fuga

Se importan las funciones de `src/pipeline.py`; no se duplica preparación en notebooks.

In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
from src.datos import DATA, DOCS, EVIDENCE
from src.datos import execute
HAS_DATA = (DATA / "obt.parquet").exists()
display(Markdown("**OBT real local disponible**" if HAS_DATA else
    "**PENDIENTE:** faltan credenciales Snowflake válidas y snapshot. Este notebook no contiene resultados empíricos del ECU 911."))


## Derivaciones y momento de inferencia

|Feature|Información utilizada|Motivo / evidencia|
|---|---|---|
|tendencia_7_28|(media7+1)/(media28+1), ambas previas a t|Cambio relativo de nivel; contrastar heterogeneidad en EDA|
|cambio_semanal|(n_t-lag7)/(lag7+1)|Variación semanal; verificar patrón EDA|
|estacional_relativa|(media de 4 weekdays+1)/(media28+1)|Ajustar señal estacional a escala local|
|sin/cos de weekday/mes objetivo|Calendario conocido|Ciclos semanales y anuales|

El histórico mensual permite una evaluación retrospectiva condicionada a disponibilidad diaria. **No demuestra que n_t estuviera accesible al cierre real t.** El uso operacional requiere un feed diario. Al ejecutar la EDA se obtiene evidencia sobre patrón semanal y escalas; la ablación posterior mide aporte de los ratios.

In [ ]:
from src.pipeline import add_history, model_input, make_pipeline, FeatureBuilder, NUMERIC, CATEGORICAL
from src.datos import load_snapshot
from sklearn.linear_model import LogisticRegression
if HAS_DATA:
    from src.datos import context
    ctx = context()
    x = model_input(ctx["train"], 3)
    features = FeatureBuilder(horizon=3).fit_transform(x)
    display(features.head())
    assert not any(c.startswith("obj_") for c in features.columns)
    print("Numéricas:", NUMERIC)
    print("Categóricas one-hot:", CATEGORICAL)
pipe = make_pipeline(LogisticRegression(random_state=42), horizon=3)
display(pipe)

## Invariantes

Para t+3, las cuatro ocurrencias históricas del weekday objetivo son t−4,−11,−18,−25. Ninguna excede t. La mediana, el one-hot (unknown=ignore) y el escalado se ajustan solo al train del fold. `obj_*` sirve únicamente para etiquetas. P90 proviene del primer año de train, congelado antes de CV. La definición no se ajusta al hold-out.

In [ ]:
if HAS_DATA:
    from src.datos import label
    y, q = label(ctx["train"], ctx["p90"], 3)
    a, b = ctx["folds"][0]
    preparation = pipe[:-1].fit(x.iloc[a], y[a])
    transformed = preparation.transform(x.iloc[b])
    print("Dimensiones validación transformada:", transformed.shape)
    print("Fecha máxima train:", ctx["train"].iloc[a].fecha.max())
    print("Fecha mínima validación:", ctx["train"].iloc[b].fecha.min())